In [1]:
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel
from langgraph.types import interrupt,Command
from langgraph.graph import START,END,StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal
from langchain_groq import ChatGroq
import json

llm = ChatGroq(model="openai/gpt-oss-20b")


movies = {
    "M001": {
        "name": "Mission Impossible",
        "show_time": "2026-07-20 18:00",
        "available_seats": 50
    },
    "M002": {
        "name": "Jurassic World",
        "show_time": "2026-07-20 21:00",
        "available_seats": 30
    },
    "M003": {
        "name": "Superman",
        "show_time": "2026-07-21 15:00",
        "available_seats": 75
    }
}

bookings = {
    "USR001": [
        {
            "booking_id": "BK101",
            "movie_id": "M001",
            "seats": 2,
            "booking_time": "2026-07-20 12:00"
        },
        {
            "booking_id": "BK102",
            "movie_id": "M002",
            "seats": 4,
            "booking_time": "2026-07-20 15:00"
        }
    ],
    "USR002": [
        {
            "booking_id": "BK103",
            "movie_id": "M003",
            "seats": 1,
            "booking_time": "2026-07-20 10:30"
        }
    ]
}


class State(BaseModel):
    query:str = ""
    action:str = ""
    human_feedback:str = ""
    user_id:str = ""
    movie_id:str = ""
    final_state:str=""


def know_intent(state:State):
    
    res = llm.invoke(
        [
            {
                "role": "system",
                "content": "You are an AI assistant designed to extract intents and entities from movie booking system queries. Your sole task is to analyze the user input and output a single JSON object matching the schema below. Do not include any explanations, introductory text, or markdown code blocks (like ```json).\n\nAllowed Intents:\n- \"booking\": User wants to book movie tickets.\n- \"cancel\": User wants to cancel an existing booking.\n- \"booking_info\": User is asking about their booking status or show details.\n\nEntities to Extract:\n- \"user_id\": Unique user identifier (null if not mentioned).\n- \"movie_name\": Title of the movie (null if not mentioned).\n\nOutput Schema:\n{\n  \"intent\": \"booking\" | \"cancel\" | \"booking_info\",\n  \"user_id\": string | null,\n  \"movie_name\": string | null\n}"},
            {
                "role": "user",
                "content": f"{state.query}"
            }
        ])
    
    values = json.loads(res.content)
    state.action = values["intent"]
    state.user_id = values["user_id"]
    movie_to_find = (values.get("movie_name") or "").lower()
    movie_id = next((k for k, v in movies.items() if v["name"].lower() == movie_to_find), "")
    state.movie_id = movie_id
    print(values)



def cancel_review_node(state:State):
    feedback = interrupt(
        {
            "question":"reply 'select specifc movie to cancel'"
        }
    )
    movie_to_find = feedback.lower()
    movie_id = next((k for k, v in movies.items() if v["name"].lower() == movie_to_find), '')
    state.movie_id = movie_id
    return state

def booking_info(state:State):
    state.final_state = movies
    return state

def cancel(state:State):
    state.final_state = "cancelled"
    return state

def booking(state:State):
    state.final_state = "booked"
    return state

def router_after_intent(state:State)->Literal["booking_info","cancel","booking","cancel_review"]:
    if state.action in ("booking", "booking_info"):
        return state.action

    print("came to router after intent")

    user_bookings = bookings.get(state.user_id, [])
    has_multiple = len(user_bookings) > 1    
    print(f"came to router after intent {user_bookings}")

    if(has_multiple):
        return "cancel_review"
    
    return "cancel"


builder = StateGraph(State)

builder.add_node("know_intent",know_intent)
builder.add_node("cancel_review",cancel_review_node)
builder.add_node("booking_info",booking_info)
builder.add_node("cancel",cancel)
builder.add_node("booking",booking)

builder.add_edge(START,"know_intent")
builder.add_conditional_edges("know_intent",router_after_intent)

builder.add_edge("booking_info",END)
builder.add_edge("booking",END)
builder.add_edge("cancel",END)
builder.add_edge("cancel_review","cancel")


graph = builder.compile(checkpointer=InMemorySaver())
conf = {"configurable":{"thread_id":1}}










# input = State()
# input.query = "my user id is USR001 i want cancel"
# know_intent(input)



/Users/sriharsha/Desktop/vector1/.venv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ModuleNotFoundError: No module named 'langchain_groq'

In [58]:
response = graph.invoke({"query":"my user id is USR001 i want cancel"},config=conf)

print(response)

{'intent': 'cancel', 'user_id': 'USR001', 'movie_name': None}
came to router after intent
came to router after intent False
{'query': 'my user id is USR001 i want cancel', 'action': '', 'human_feedback': '', 'user_id': '', 'movie_id': '', 'final_state': 'cancelled'}
